# pyspi

pyspi computes **statistics of pairwise interactions** (SPIs) between the processes of a multivariate time series. Each SPI is one answer to "how are these two processes related?" — a correlation, a spectral coherence, a transfer entropy, a causal score. For `M` processes each returns an `M x M` matrix, and pyspi stacks them so hundreds of notions of "related" can be compared side by side.

## 1. Data

In [1]:
import numpy as np
from pyspi.calculator import Calculator, bundled_configs
from pyspi.data import load_dataset, available_datasets

available_datasets()

{'forex': 'Foreign-exchange rates (250 obs, 7 processes).',
 'cml': 'Coupled map lattice (500 obs, 10 processes).',
 'standard_normal': 'i.i.d. standard normal null model (200 obs, 5 processes).'}

`Data` holds an `M x T` array, z-scored per process by default. Your own data goes in the same way — `Calculator(dataset=my_array)` accepts a NumPy array, and `dim_order` controls whether rows are processes or observations.

In [2]:
data = load_dataset("cml")   # coupled map lattice: 10 processes, 500 observations
print(data.n_processes, "processes x", data.n_observations, "observations")

10 processes x 500 observations


## 2. Compute

`config` selects the SPI set. `fabfour` is the smallest (4 SPIs); `full` has all 231. See `bundled_configs()`.

In [3]:
calc = Calculator(dataset=data, config="benchmarked_p90", verbose=False)
calc.compute(progress=False)

calc.table

spi     cov_EmpiricalCovariance                                          \
process                  proc-0    proc-1    proc-2    proc-3    proc-4   
proc-0                      NaN  0.421441  0.027314  0.000476 -0.018337   
proc-1                 0.421441       NaN  0.709196  0.336793  0.172608   
proc-2                 0.027314  0.709196       NaN  0.722390  0.442336   
proc-3                 0.000476  0.336793  0.722390       NaN  0.847342   
proc-4                -0.018337  0.172608  0.442336  0.847342       NaN   
proc-5                -0.015935 -0.149192 -0.147153 -0.091264  0.155771   
proc-6                 0.002972 -0.163586 -0.212502 -0.268826 -0.166982   
proc-7                 0.004477 -0.167011 -0.223507 -0.255886 -0.201453   
proc-8                -0.027090 -0.078471 -0.105765 -0.010023 -0.005019   
proc-9                 0.019078 -0.114243 -0.131392 -0.031943 -0.062203   

spi                                                        ...       ids  \
process    proc-5    proc-6    proc-7    proc-8    proc-9  ...    proc-0   
proc-0  -0.015935  0.002972  0.004477 -0.027090  0.019078  ...       NaN   
proc-1  -0.149192 -0.163586 -0.167011 -0.078471 -0.114243  ...  0.523740   
proc-2  -0.147153 -0.212502 -0.223507 -0.105765 -0.131392  ...  0.052004   
proc-3  -0.091264 -0.268826 -0.255886 -0.010023 -0.031943  ...  0.033250   
proc-4   0.155771 -0.166982 -0.201453 -0.005019 -0.062203  ...  0.035241   
proc-5        NaN  0.845025  0.671742 -0.164250 -0.114857  ...  0.037163   
proc-6   0.845025       NaN  0.881547 -0.193940 -0.051837  ...  0.023086   
proc-7   0.671742  0.881547       NaN -0.112729  0.206975  ...  0.033665   
proc-8  -0.164250 -0.193940 -0.112729       NaN -0.021290  ...  0.040405   
proc-9  -0.114857 -0.051837  0.206975 -0.021290       NaN  ...  0.049165   

spi                                                                            \
process    proc-1    proc-2    proc-3    proc-4    proc-5    proc-6    proc-7   
proc-0   0.523740  0.052004  0.033250  0.035241  0.037163  0.023086  0.033665   
proc-1        NaN  0.664840  0.377530  0.236184  0.206985  0.182929  0.174328   
proc-2   0.664840       NaN  0.804516  0.476518  0.251158  0.217836  0.211789   
proc-3   0.377530  0.804516       NaN  0.862172  0.290385  0.289384  0.257373   
proc-4   0.236184  0.476518  0.862172       NaN  0.646588  0.402762  0.366253   
proc-5   0.206985  0.251158  0.290385  0.646588       NaN  0.815337  0.680093   
proc-6   0.182929  0.217836  0.289384  0.402762  0.815337       NaN  0.864102   
proc-7   0.174328  0.211789  0.257373  0.366253  0.680093  0.864102       NaN   
proc-8   0.136872  0.172217  0.131387  0.139737  0.178159  0.246189  0.351529   
proc-9   0.104155  0.150584  0.174069  0.205814  0.255597  0.271616  0.438352   

spi                          
process    proc-8    proc-9  
proc-0   0.040405  0.049165  
proc-1   0.136872  0.104155  
proc-2   0.172217  0.150584  
proc-3   0.131387  0.174069  
proc-4   0.139737  0.205814  
proc-5   0.178159  0.255597  
proc-6   0.246189  0.271616  
proc-7   0.351529  0.438352  
proc-8        NaN  0.484431  
proc-9   0.484431       NaN  

[10 rows x 1610 columns]

## 3. Reading the results

`calc` itself shows what happened. `calc.table` is wide: rows are processes, columns a `(spi, process)` MultiIndex, shape `M x (n_spis * M)`.

In [4]:
calc.summary()

{'dataset': 'N/A',
 'n_processes': 10,
 'n_observations': 500,
 'config': 'benchmarked_p90',
 'n_spis': 161,
 'n_computed': 161,
 'n_failed': 0,
 'failed': [],
 'n_not_appropriate': 0,
 'not_appropriate': [],
 'total_seconds': 15.967,
 'slowest': [('lcss', 3.205),
  ('bary_dtw_corr', 2.207),
  ('dcoh_multitaper_mean_fs-1_fmin-0_fmax-0-5', 1.631),
  ('cds', 0.82),
  ('sgc_nonparametric_mean_fs-1_fmin-1e-05_fmax-0-5', 0.804)]}

In [5]:
print(calc.table.shape, calc.table.columns.names)
calc.table.iloc[:3, :6].round(3)

(10, 1610) ['spi', 'process']


spi     cov_EmpiricalCovariance                                   
process                  proc-0 proc-1 proc-2 proc-3 proc-4 proc-5
proc-0                      NaN  0.421  0.027  0.000 -0.018 -0.016
proc-1                    0.421    NaN  0.709  0.337  0.173 -0.149
proc-2                    0.027  0.709    NaN  0.722  0.442 -0.147

### One SPI

Indexing by identifier returns that SPI's `M x M` matrix. Entry `[i, j]` is computed with `i` as **source**, `j` as **target** — irrelevant for a symmetric SPI, the whole point for a directed one. The diagonal is NaN: self-pairs are not computed.

In [6]:
cov = calc.table["cov_EmpiricalCovariance"]
cov.round(3)

process,proc-0,proc-1,proc-2,proc-3,proc-4,proc-5,proc-6,proc-7,proc-8,proc-9
proc-0,NaN,0.421,0.027,0.000,-0.018,-0.016,0.003,0.004,-0.027,0.019
proc-1,0.421,NaN,0.709,0.337,0.173,-0.149,-0.164,-0.167,-0.078,-0.114
proc-2,0.027,0.709,NaN,0.722,0.442,-0.147,-0.213,-0.224,-0.106,-0.131
proc-3,0.000,0.337,0.722,NaN,0.847,-0.091,-0.269,-0.256,-0.010,-0.032
proc-4,-0.018,0.173,0.442,0.847,NaN,0.156,-0.167,-0.201,-0.005,-0.062
proc-5,-0.016,-0.149,-0.147,-0.091,0.156,NaN,0.845,0.672,-0.164,-0.115
proc-6,0.003,-0.164,-0.213,-0.269,-0.167,0.845,NaN,0.882,-0.194,-0.052
proc-7,0.004,-0.167,-0.224,-0.256,-0.201,0.672,0.882,NaN,-0.113,0.207
proc-8,-0.027,-0.078,-0.106,-0.010,-0.005,-0.164,-0.194,-0.113,NaN,-0.021
proc-9,0.019,-0.114,-0.131,-0.032,-0.062,-0.115,-0.052,0.207,-0.021,NaN


In [7]:
# a single pair, and the strongest pairs overall
print("proc-0 / proc-3:", round(cov.iloc[0, 3], 4))

upper = cov.where(np.triu(np.ones(cov.shape), k=1).astype(bool))
upper.stack().abs().sort_values(ascending=False).head(5).round(3)

proc-0 / proc-3: 0.0005


        process
proc-6  proc-7     0.882
proc-3  proc-4     0.847
proc-5  proc-6     0.845
proc-2  proc-3     0.722
proc-1  proc-2     0.709
dtype: float64

### Direction

The data is z-scored, so covariance *is* Pearson correlation and is symmetric. Directed information is not — the two triangles are separate estimates.

In [8]:
di = calc.table["di_gaussian_n-5"]
print("0 -> 1:", round(di.iloc[0, 1], 3), " | 1 -> 0:", round(di.iloc[1, 0], 3))
print("di symmetric?", np.allclose(di.values, di.values.T, equal_nan=True),
      "| cov symmetric?", np.allclose(cov.values, cov.values.T, equal_nan=True))

0 -> 1: 0.862  | 1 -> 0: 0.3
di symmetric? False | cov symmetric? True


### Long form

`to_frame()` gives one row per `(spi, source, target)` — the shape you want for plotting, `groupby`, or joining against SPI labels.

In [9]:
long = calc.to_frame()
display(long.head(3))
long.groupby("spi")["value"].agg(["mean", "std", "count"]).round(3)

,spi,source,target,value
0,cov_EmpiricalCovariance,proc-0,proc-1,0.421441
1,cov_EmpiricalCovariance,proc-0,proc-2,0.027314
2,cov_EmpiricalCovariance,proc-0,proc-3,0.000476


,mean,std,count
spi,,,
bary_dtw_corr,0.612,0.238,90
cce_gaussian_n-5,4.211,1.465,90
cce_kozachenko_n-5,-1.560,5.078,90
cds,0.067,0.028,90
ce_kernel_W-0.5,1.025,0.212,90
...,...,...,...
xme_kernel_W-0.5_k1,1.023,0.229,90
xme_kozachenko_k1,0.248,1.555,90
xme_kozachenko_k10,6.606,11.411,90


## 4. Cost

Cost is heavily skewed: a few SPIs dominate, most are effectively free. `summary()` reports the total and the slowest five, which is how to budget a run.

The `benchmarked_p80/p90/p95/p99` configs exploit that skew — they keep the fastest N% of the SPIs timed in pyspi's compute-cost benchmark. SPIs added to `full` after that benchmark are in none of them. To build your own subset by keyword, `filter_spis(["directed", "nonlinear"], output_name="mine")` writes a config you pass as `Calculator(config="mine.yaml")`.

In [10]:
import time

t0 = time.perf_counter()
sonnet = Calculator(dataset=data, config="sonnet", verbose=False)   # 14 SPIs, about one per module
sonnet.compute(progress=False)
print(f"{sonnet.n_spis} SPIs in {time.perf_counter() - t0:.1f}s")

sonnet.summary()["slowest"]

14 SPIs in 17.4s


[('te_kraskov_NN-4_DCE-AUTO_MAX-CORR-AIS_k-max-10_tau-max-4', 14.298),
 ('bary_dtw_corr', 1.497),
 ('sgc_nonparametric_mean_fs-1_fmin-1e-05_fmax-0-5', 0.816),
 ('coint_aeg_tstat_trend-ct_autolag-aic_maxlag-10', 0.315),
 ('anm', 0.259)]

## 5. Failures

A failed SPI still occupies its column, filled with NaN, with its message in `calc.errors`. An SPI that is not appropriate for the data (too few observations, tied values) is NaN too, with its reason in `calc.not_appropriate`; it has not failed. Check both before reading a table: they say why a NaN column is NaN.

In [11]:
sonnet.errors or "no failures"

'no failures'

## 6. Inspecting the calculator

What was run, what it cost, and what is available.

In [12]:
print("SPIs:", calc.n_spis)
print("first five:", list(calc.spis)[:5])

# every parameter that defines this run -- what checkpoints are bound to
{k: v for k, v in calc.run_spec.items() if k not in ("spi_identifiers", "configfile")}

SPIs: 161
first five: ['cov_EmpiricalCovariance', 'cov_MinCovDet', 'pcorr_EmpiricalCovariance', 'pcorr_GraphicalLasso', 'pcorr_LedoitWolf']


{'config': 'benchmarked_p90',
 'zscore': True,
 'detrend': False,
 'n_processes': 10,
 'n_observations': 500,
 'procnames': ['proc-0',
  'proc-1',
  'proc-2',
  'proc-3',
  'proc-4',
  'proc-5',
  'proc-6',
  'proc-7',
  'proc-8',
  'proc-9'],
 'dataset_name': 'N/A',
 'computation_version': '3.0.0.r15',
 'pyspi_version': '3.0.0',
 'dependency_versions': {'colorama': '0.4.6',
  'dtaidistance': '2.4.0',
  'hyppo': '0.5.2',
  'mne': '1.12.1',
  'mne-connectivity': '0.8.1',
  'nitime': '0.12.1',
  'numba': '0.63.1',
  'numpy': '2.3.5',
  'pandas': '2.3.3',
  'pyEDM': '2.5.0',
  'pyyaml': '6.0.3',
  'scikit-learn': '1.9.1',
  'scipy': '1.17.1',
  'spectral-connectivity': '2.0.0',
  'statsmodels': '0.14.6',
  'tqdm': '4.67.3',
  'tslearn': '0.10.0'}}

In [13]:
# labels drive filtering; each SPI carries the traits it actually has
spi = calc.spis["di_gaussian_n-5"]
print(spi.identifier, "->", sorted(spi.labels))

# build your own subset by keyword
from pyspi.utils import filter_spis
# filter_spis(["directed", "nonlinear"], output_name="mine")   # -> mine.yaml

di_gaussian_n-5 -> ['M03', 'bivariate', 'directed', 'infotheory', 'linear', 'temporal', 'time-dependent', 'unsigned']


## 7. Saving, scale, CLI

`.npz` round-trips exactly and needs nothing but numpy; `.csv` is a one-way export.

```python
calc.save("results.npz")
from pyspi.calculator import load_table
table = load_table("results.npz")
```

**Many datasets: run one per process** (a cluster array job, or GNU parallel) rather than raising `n_jobs`. SPIs sharing a cache are grouped into one sequential task, so within-dataset speedup is floored by the longest group — measured at 2.3–4.5x whatever `n_jobs` you pass. One dataset per process scales close to linearly and confines a failure to one dataset.

Everything above is available without writing Python:

```bash
python -m pyspi compute --data ts.npy --config fabfour --checkpoint-dir results/
```

`--checkpoint-dir` writes each SPI as it finishes, so an interrupted run resumes instead of restarting.